# AL6 — Panorama Building: Stitching a 1958 Flight Block
**MCS 3950 Computer Vision · UPEI · Fall 2026 · Friday Oct 23**

L8 stitched **two** photographs: one homography, one canvas, one feathered blend. A real
panorama, or a real air-photo mosaic, has many photographs, and nobody tells you which
ones overlap. Today you build the whole pipeline that phone panoramas and Brown & Lowe's
*AutoStitch* (2007) use, on eleven 1958 air photos from two neighbouring flight lines:

1. **match every pair** and decide which pairs really overlap;
2. **chain** the pairwise homographies into one reference frame;
3. **blend** everything onto one canvas;
4. **close the loop**: make all the overlaps agree at once.

Flat farmland seen from 3,000 m is a plane, so every pair of frames is related by a
homography (L8 §2.2, case 1). A phone panorama gets its homographies from a camera
that only rotates (case 2). The pipeline is the same.

| Part | Topic | Time |
|------|-------|------|
| 1 | Which frames overlap? Matching all 55 pairs | ~12 min |
| 2 | One strip: chaining homographies and blending | ~15 min |
| 3 | Two strips: closing the loop | ~20 min |
| ★ | Bonus: what `cv2.Stitcher` does differently | if time |

Each part has a **task** (four in all) and **checkpoint questions**.

> **Not submitted or graded.** This is Test 2 material (L7–L11). **Project 2** (aerial
> mosaic) groups: Part 3 is your project in miniature. Everyone else who needs to put two
> eras or two frames on one grid will meet the same problems.

In [ ]:
import io, os, time, urllib.request, zipfile
import cv2
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection
from scipy.optimize import least_squares

plt.rcParams.update({"figure.dpi": 100, "axes.spines.top": False,
                     "axes.spines.right": False, "font.size": 11})
GOLD, RED, TEAL, GREEN, NAVY, GREY = "#C8962C", "#B82828", "#1A7A8A", "#2E7D50", "#1A3A5C", "#9AA5B1"

def show(images, titles, cmap='gray', vmin=0, vmax=255, figw=15, figh=4.5):
    # Show a row of images (BGR colour images are converted for display).
    fig, axes = plt.subplots(1, len(images), figsize=(figw, figh))
    if len(images) == 1:
        axes = [axes]
    for ax, im, t in zip(axes, images, titles):
        if im.ndim == 3:
            ax.imshow(cv2.cvtColor(im, cv2.COLOR_BGR2RGB))
        else:
            ax.imshow(im, cmap=cmap, vmin=vmin, vmax=vmax)
        ax.set_title(t, fontsize=11); ax.axis('off')
    plt.tight_layout(); plt.show()

def project(p, H):
    # Apply a 3x3 homography to an (N, 2) array of points: lift, multiply, divide.
    q = np.c_[p, np.ones(len(p))] @ H.T
    return q[:, :2] / q[:, 2:]

print(f"OpenCV {cv2.__version__}")

## Get the imagery

The same `AL5.zip` as AL5 and L8 (~8 MB). Today we use its eleven frames from flight
line 16046 and ignore the nine distractors. Safe to re-run.

In [ ]:
BASE = ("https://raw.githubusercontent.com/andrewgodbout/"
        "MCS-3950-F26/main/Notebooks/data/")

def fetch_zip(name, needed):
    if all(os.path.exists(f) for f in needed):
        print(f"{name}: already present")
        return
    with urllib.request.urlopen(BASE + name, timeout=120) as r:
        blob = r.read()
    with zipfile.ZipFile(io.BytesIO(blob)) as z:
        z.extractall(".")
    print(f"{name}: {len(blob)/1e6:.1f} MB downloaded and extracted")

fetch_zip("AL5.zip", ["frames/16046-19.jpg", "frames/16046-20.jpg"])

NAMES = [f"16046-{k}" for k in (17, 18, 19, 20, 21, 22, 101, 102, 103, 104, 105)]
FRAMES = [cv2.imread(f"frames/{n}.jpg", cv2.IMREAD_GRAYSCALE) for n in NAMES]
IDX = {n: i for i, n in enumerate(NAMES)}                  # "16046-20" -> 3
STRIP1, STRIP2 = list(range(0, 6)), list(range(6, 11))     # frame indices on each line
REF = IDX["16046-20"]                                       # the mosaic lives in frame 20's pixels

# L8's reference homography 19 -> 20 (fitted on the full-resolution scans). Used only to score.
H_REF = np.array([[ 0.98700935, -0.00873884, -390.04915793],
                  [ 0.00098735,  1.01974419,  -25.81833349],
                  [-0.00000007,  0.00002919,    1.        ]])

fig, axes = plt.subplots(2, 6, figsize=(16, 6))
for ax in axes.ravel():
    ax.axis('off')
for row, strip in enumerate((STRIP1, STRIP2)):
    for col, i in enumerate(strip):
        axes[row, col].imshow(FRAMES[i], cmap='gray'); axes[row, col].set_title(NAMES[i], fontsize=10)
plt.suptitle("Eleven 1958 frames from flight line 16046 (about 3.5 m per pixel, 1000 px tall)", y=0.98)
plt.tight_layout(); plt.show()

---
## Part 1 — Which frames overlap?  `⏱ ~12 min`

A phone that stitches a panorama is handed a pile of photos. It does not know the order,
so it matches **every pair** and lets geometry decide which pairs are real. Eleven frames
make $\binom{11}{2} = 55$ pairs.

### Step 1 · Features, with a budget

Each frame has about 11,000 SIFT keypoints. Brute-force matching of 55 pairs at that size
takes about 1½ minutes. Capping SIFT at the **3,000 strongest** keypoints per frame cuts
that to a few seconds, and on the L8 pair (19 → 20) the capped fit still lands within a
pixel of `H_REF` (checked below).

In [ ]:
sift = cv2.SIFT_create(nfeatures=3000)
t0 = time.perf_counter()
FEAT = [sift.detectAndCompute(im, None) for im in FRAMES]
KPT = [np.float64([k.pt for k in kps]) for kps, _ in FEAT]      # (N, 2) keypoint positions
DESC = [d for _, d in FEAT]
print(f"SIFT on {len(FRAMES)} frames: {time.perf_counter() - t0:.1f} s, "
      f"{min(len(k) for k in KPT):,}–{max(len(k) for k in KPT):,} keypoints per frame")

### Step 2 · Match one pair, and decide whether to believe it  (Task 1)

Everything here is from L8: ratio-test candidates, a robust homography, and then the
question L8 §11 ended on: **RANSAC always returns a model, so is this one real?**
Brown & Lowe's test accepts a pair only if

$$n_\text{inliers} \;>\; 8 + 0.3\, n_\text{candidates}.$$

**Task 1:** complete `match_pair`. `match_pair` takes as input two integers i and j representing the frames that we want to find matches within, a value r which is the SIFT ratio test and then an epsilon eps which is the value to use in the Brown and Lowe test.

Matches come from SIFT features that pass the ratio test i.e., $d1/d2 < r$ or $d1 < r*d2$. `BF.knnMatch(des1, des2, k=2)` returns list of ordered lists (length=k) of `DMatch` where DMatch has the following attributes:

  - distance the distance between the two features
  - queryIdx the index of the descriptor in the first frame (des1)
  - trainIdx the index of hte descriptor in the 2nd frame (des2)

We will use the distance to keep only those that pass the ratio test and then use the Idx's to reference the actual positions in the frames.

Those positions can be passed to `H, mask = cv2.findHomography(p, q, cv2.USAC_MAGSAC, eps)`
which will select samples of size 4 and use that to compute a homography and then returns the homography that produced the most inliers. The method returns that best homography H and a mask giving 0 for outlier and 1 for inlier e.g,[ [1], ..., [0]]  

Use that mask to count the inliers and determine if it passes the Brown & Lowe test and also to determine which inliers to return (the one's marked 1).


**Note:** Read Step 1 closely,

**DESC** is a list with 11 entries. Each entry in that list is a 2D np array with each row of the array being a 128 floating point values representing the SIFT feature. There may be a few more than 3000 rows in this array since OpenCV will retain ties.

So `DESC[i]` is a numpy 2D array with at least 3000 rows, each row 128 values (representing a SIFT feature) giving all descriptors for frame i, and DESC[i][k] is the kth feature descriptor associated with frame i.

**KPT** is a list with the pixel positions of those descriptors.

`KPT[i][k]` is the (x,y) position of k'th feature descriptor found at DESC[i][k] from image i.  


In [ ]:
BF = cv2.BFMatcher(cv2.NORM_L2)

def match_pair(i, j, r=0.8, eps=3.0):
    '''
    Match frame i to frame j.

    Returns a dict with
      n_cand : number of ratio-test candidates
      n_inl  : number of RANSAC inliers
      H      : 3x3 homography i -> j (None if there were fewer than 4 candidates)
      p, q   : (n_inl, 2) inlier positions in frame i and frame j
      ok     : True if Brown & Lowe's test accepts the pair
    '''

    # ── YOUR CODE HERE ──────────────────────────────────────────────────────────
    # 1. knn = BF.knnMatch(DESC[i], DESC[j], k=2); keep m where m.distance < r * n.distance
    # 2. p = KPT[i][m.queryIdx], q = KPT[j][m.trainIdx] for the kept matches
    # 3. if fewer than 4 candidates: return n_inl=0, H=None, empty p and q, ok=False
    # 4. H, mask = cv2.findHomography(p, q, cv2.USAC_MAGSAC, eps); inl = mask.ravel().astype(bool)
    # 5. ok = n_inl > 8 + 0.3 * n_cand
    # ────────────────────────────────────────────────────────────────────────────
    raise NotImplementedError("Task 1: match the pair and test it.")

In [ ]:
# Check on L8's pair, 19 -> 20, and on a pair with no overlap, 17 -> 20.
_h19, _w19 = FRAMES[IDX["16046-19"]].shape; _h20, _w20 = FRAMES[REF].shape
yy, xx = np.mgrid[0:_h19:10, 0:_w19:10]
_grid = np.c_[xx.ravel(), yy.ravel()].astype(float)
_g20 = project(_grid, H_REF)
_inside = np.all((_g20 >= 0) & (_g20 < [_w20, _h20]), axis=1)
GRID19 = _grid[_inside]                                    # frame-19 points that land in frame 20

def err_vs_ref(H19_20):
    # Mean distance (px) between where H and H_REF send frame 19's overlap with frame 20.
    return np.linalg.norm(project(GRID19, H19_20) - project(GRID19, H_REF), axis=1).mean()

try:
    for a, b in (("16046-19", "16046-20"), ("16046-17", "16046-20")):
        m = match_pair(IDX[a], IDX[b])
        extra = f",  {err_vs_ref(m['H']):.2f} px from H_REF" if a == "16046-19" else ""
        print(f"{a} -> {b}: {m['n_cand']:4} candidates, {m['n_inl']:4} inliers, "
              f"needs > {8 + 0.3*m['n_cand']:4.0f}  ->  {'ACCEPT' if m['ok'] else 'reject'}{extra}")
except NotImplementedError:
    print("Complete Task 1 first.")

### Step 3 · All 55 pairs

Each pair is matched once, as `i → j` with `i < j`. If you need `j → i` later, invert
`H` (`H_between` below does that for you).

In [ ]:
PAIRS = {}
try:
    t0 = time.perf_counter()
    for i in range(len(NAMES)):
        for j in range(i + 1, len(NAMES)):
            PAIRS[(i, j)] = match_pair(i, j)
    print(f"55 pairs in {time.perf_counter() - t0:.1f} s; "
          f"{sum(v['ok'] for v in PAIRS.values())} accepted")
except NotImplementedError:
    print("Complete Task 1 first.")

def H_between(i, j):
    # Homography frame i -> frame j, from whichever direction we matched.
    return PAIRS[(i, j)]["H"] if (i, j) in PAIRS else np.linalg.inv(PAIRS[(j, i)]["H"])

def accepted(i, j):
    return PAIRS[(min(i, j), max(i, j))]["ok"]

In [ ]:
if PAIRS:
    n = len(NAMES)
    M = np.zeros((n, n)); OK = np.zeros((n, n), bool)
    for (i, j), v in PAIRS.items():
        M[i, j] = M[j, i] = v["n_inl"]; OK[i, j] = OK[j, i] = v["ok"]

    fig, ax = plt.subplots(1, 2, figsize=(16, 6.4), gridspec_kw={"width_ratios": [1.15, 1]})
    im = ax[0].imshow(np.where(OK, M, np.nan), cmap="viridis")
    for i in range(n):
        for j in range(n):
            if i != j:
                ax[0].text(j, i, f"{int(M[i, j])}", ha="center", va="center", fontsize=8,
                           color="white" if OK[i, j] else GREY)
    short = [s.split("-")[1] for s in NAMES]
    ax[0].set_xticks(range(n), short); ax[0].set_yticks(range(n), short)
    ax[0].axhline(5.5, c="k", lw=1); ax[0].axvline(5.5, c="k", lw=1)
    ax[0].set_title("RANSAC inliers per pair (coloured = accepted, grey = rejected)")
    plt.colorbar(im, ax=ax[0], fraction=0.046, label="inliers")

    c = np.array([v["n_cand"] for v in PAIRS.values()]); k = np.array([v["n_inl"] for v in PAIRS.values()])
    ok = np.array([v["ok"] for v in PAIRS.values()])
    ax[1].scatter(c[ok], k[ok], s=40, c=TEAL, label=f"accepted ({ok.sum()})")
    ax[1].scatter(c[~ok], k[~ok], s=40, c=RED, marker="x", label=f"rejected ({(~ok).sum()})")
    xs = np.linspace(40, c.max() * 1.05, 50)
    ax[1].plot(xs, 8 + 0.3 * xs, c=NAVY, ls="--", lw=2, label="8 + 0.3 × candidates")
    ax[1].set_xscale("log"); ax[1].set_yscale("log")
    ax[1].set_xlabel("ratio-test candidates"); ax[1].set_ylabel("RANSAC inliers")
    ax[1].set_title("Every pair: real overlaps and coincidences separate cleanly")
    ax[1].legend(loc="upper left"); ax[1].grid(alpha=0.3, which="both")
    plt.tight_layout(); plt.show()

    rej = [(NAMES[i], NAMES[j], v["n_cand"], v["n_inl"]) for (i, j), v in PAIRS.items() if not v["ok"]]
    print(f"rejected pairs: {len(c) - ok.sum()}, candidates {min(r[2] for r in rej)}–{max(r[2] for r in rej)}, "
          f"inliers {min(r[3] for r in rej)}–{max(r[3] for r in rej)}")
    best_rej = max(rej, key=lambda r: r[3] - 0.3 * r[2])
    print(f"closest call: {best_rej[0]} -> {best_rej[1]}: {best_rej[2]} candidates, {best_rej[3]} inliers "
          f"(needed > {8 + 0.3*best_rej[2]:.0f})")

### Checkpoint 1

**Q1a.** Read the layout off the matrix. On one flight line, how many frames apart
can two photos be and still overlap? Which frame on the second line (101–105) overlaps
**16046-19** most? Sketch the eleven frames as rectangles: two rows, in the right order.

*Your answer:*

**Q1b.** The rejected pairs still have 100+ ratio-test candidates and a handful of
RANSAC "inliers". Where do those inliers come from (L8 §11)? Why does the acceptance
threshold grow with the number of candidates instead of being a fixed number of inliers?

*Your answer:*

---
## Part 2 — One strip  `⏱ ~15 min`

### Step 1 · Chaining homographies  (Task 2)

We want every frame in **one** coordinate system: frame 20's pixels. Let $G_i$ be the
homography that takes frame $i$ into frame 20, so $G_{20} = I$ (Indentity). Frame 19 has a direct
match: $G_{19} = H_{19\to20}$. Frame 18 does not need one. Go 18 → 19 → 20:

$$G_{18} = G_{19}\,H_{18\to19}, \qquad G_{17} = G_{18}\,H_{17\to18}, \qquad
G_{21} = G_{20}\,H_{21\to20}, \;\dots$$

Read the right-hand side from right to left: first into the neighbour, then on from the
neighbour into 20. **Order matters**: $H_{18\to19}\,G_{19}$ is a different (wrong) matrix.

**Task 2:** complete `chain(order, ref)`. `order` lists frame indices along the strip; walk
outwards from `ref` in both directions and return a dict `{frame index: G}`.

In [ ]:
def chain(order, ref):
    '''
    order : frame indices in strip order, e.g. STRIP1 = [0, 1, 2, 3, 4, 5]
    ref   : the frame whose pixels the mosaic uses (must be in order)

    Returns {i: G_i}, where G_i maps frame i into frame ref and G_ref = I.
    Use H_between(i, j) for the homography frame i -> frame j.
    '''
    G = {ref: np.eye(3)} # G is the dictionary to return. np.eye(3) is 3x3 identity
    k = order.index(ref)

    G = {ref: np.eye(3)}
    k = order.index(ref)
    for a in range(k - 1, -1, -1):                       # walk left
        i, nb = order[a], order[a + 1]
        G[i] = G[nb] @ H_between(i, nb)
    for a in range(k + 1, len(order)):                   # walk right
        i, nb = order[a], order[a - 1]
        G[i] = G[nb] @ H_between(i, nb)
    return G

    # ── YOUR CODE HERE ──────────────────────────────────────────────────────────
    # walk left:  for a = k-1 down to 0:        i = order[a], nb = order[a+1]
    # walk right: for a = k+1 up to len(order)-1: i = order[a], nb = order[a-1]
    # each time:  G[i] = G[nb] @ H_between(i, nb)
    # ────────────────────────────────────────────────────────────────────────────
    raise NotImplementedError("Task 2: chain the homographies.")

In [ ]:
CORNERS = lambda im: np.array([[0, 0], [im.shape[1], 0], [im.shape[1], im.shape[0]], [0, im.shape[0]]], float)

try:
    G1 = chain(STRIP1, REF)
    print(f"{'frame':>10} {'hops':>5} {'centre in frame 20 (px)':>26} {'scale':>7}")
    for i in STRIP1:
        c = project(np.array([[FRAMES[i].shape[1] / 2, FRAMES[i].shape[0] / 2]]), G1[i])[0]
        s = np.sqrt(abs(np.linalg.det(G1[i][:2, :2] / G1[i][2, 2])))
        print(f"{NAMES[i]:>10} {abs(i - REF):5} {c[0]:12.0f} {c[1]:12.0f} {s:8.3f}")
    print(f"\nG_19 equals H_19->20: {np.allclose(G1[IDX['16046-19']], H_between(IDX['16046-19'], REF))}")
except (NotImplementedError, NameError):
    print("Complete Task 2 first.")

### Step 2 · Every accepted pair is a check

The chain used only the **5** consecutive pairs. The strip has **8** accepted pairs: the
other three (17–19, 18–20, …) skip a frame, and the chain never saw them. If the chain
is right, their inliers must land on top of each other too. For any accepted pair
$(i, j)$ with inliers $p \leftrightarrow q$, the disagreement in the mosaic is

$$e = \lVert G_i\,p \;-\; G_j\,q \rVert \quad\text{(pixels of frame 20)}.$$

In [ ]:
def pair_errors(G):
    # {(i, j): (rms, max, n)} for every accepted pair whose two frames are both in G.
    out = {}
    for (i, j), v in PAIRS.items():
        if v["ok"] and i in G and j in G:
            e = np.linalg.norm(project(v["p"], G[i]) - project(v["q"], G[j]), axis=1)
            out[(i, j)] = (np.sqrt(np.mean(e**2)), e.max(), len(e))
    return out

def summarise(G, label):
    E = pair_errors(G)
    r = np.array([x[0] for x in E.values()]); n = np.array([x[2] for x in E.values()])
    (wi, wj), w = max(E.items(), key=lambda kv: kv[1][0])
    overall = np.sqrt(np.sum(r**2 * n) / n.sum())
    print(f"{label:34} {len(E):2} pairs   RMS {overall:5.2f} px   worst {NAMES[wi][6:]}–{NAMES[wj][6:]}: "
          f"{w[0]:5.1f} px RMS, {w[1]:5.1f} max")
    return overall, E

try:
    G1 = chain(STRIP1, REF)
    _, E1 = summarise(G1, "strip 1, chained into 20")
    for (i, j), (rms, mx, n_) in E1.items():
        print(f"   {NAMES[i]} – {NAMES[j]:10} {'skip' if j - i > 1 else 'used':>5}   {n_:4} inliers   "
              f"RMS {rms:4.2f} px   max {mx:4.1f} px")
except (NotImplementedError, NameError):
    print("Complete Task 2 first.")

### Step 3 · One canvas  (Task 3)

Same recipe as L8 §9.2–9.3, for any number of frames:

1. push every frame's four corners through its $G_i$; take the bounding box of them all;
2. $T$ = the translation that moves the box's top-left to $(0, 0)$;
3. warp each frame by $T\,G_i$, and warp an all-255 image the same way to get its footprint;
4. **feather**: weight each warped frame by its footprint's distance transform (far from
   its own edge = more say), add them all up, and divide by the summed weights.


In [ ]:
def mosaic(G, frames=FRAMES):
    '''
    G : {frame index: homography into the reference frame}
    Returns (mosaic as uint8, T) where T is the 3x3 shift onto the canvas.
    '''
    C = np.vstack([project(CORNERS(frames[i]), Gi) for i, Gi in G.items()])
    x0, y0 = np.floor(C.min(0)); x1, y1 = np.ceil(C.max(0))
    T = np.array([[1, 0, -x0], [0, 1, -y0], [0, 0, 1]])
    size = (int(x1 - x0), int(y1 - y0))
    assert max(size) < 6000, f"canvas {size} is far too big: is one of the G's inverted?"
    acc = np.zeros(size[::-1], np.float32); wsum = np.zeros(size[::-1], np.float32)
    for i, Gi in G.items():
        f = frames[i]
        foot = cv2.warpPerspective(np.full_like(f, 255), T @ Gi, size, flags=cv2.INTER_NEAREST)
        w = cv2.distanceTransform((foot > 0).astype(np.uint8), cv2.DIST_L2, 5)
        acc += w * cv2.warpPerspective(f, T @ Gi, size).astype(np.float32)
        wsum += w
    return (acc / np.maximum(wsum, 1e-6)).astype(np.uint8), T

In [ ]:
def outline(ax, G, T, idx, colour, lw=1.2, label=True):
    # Draw each frame's footprint on a mosaic.
    for i in idx:
        c = project(CORNERS(FRAMES[i]), T @ G[i])
        ax.plot(*np.r_[c, c[:1]].T, c=colour, lw=lw)
        if label:
            m = c.mean(0); ax.text(m[0], m[1], NAMES[i][6:], color=colour, fontsize=12,
                                   ha="center", va="center", weight="bold",
                                   bbox=dict(fc="white", alpha=0.6, lw=0, pad=1))
try:
    t0 = time.perf_counter()
    MOS1, T1 = mosaic(G1)
    print(f"strip 1: {MOS1.shape[1]} x {MOS1.shape[0]} canvas in {time.perf_counter() - t0:.1f} s")
    fig, ax = plt.subplots(figsize=(16, 6.5))
    ax.imshow(MOS1, cmap="gray"); outline(ax, G1, T1, STRIP1, GOLD); ax.axis("off")
    ax.set_title("Flight line 16046, frames 17–22, in frame 20's coordinates (feathered)")
    plt.tight_layout(); plt.show()
except (NotImplementedError, NameError):
    print("Complete Tasks 2 and 3 first.")

### Checkpoint 2

**Q2a.** Why chain into frame **20** rather than into frame 17, the first one?
Count the hops to the farthest frame each way. What happens to an error in one link as
it travels down the chain?

*Your answer:*

**Q2b.** The three *skip* pairs disagree more than the five pairs the chain used.
Why should that be expected, even if every pairwise homography is as good as it can be?
What would the skip pairs' errors do on a strip of 30 frames?

*Your answer:*

**Q2c.** Look closely at the mosaic. Name two visible artefacts that **no** amount
of better geometry would remove, and say what causes each.

*Your answer:*

---
## Part 3 — Two strips: closing the loop  `⏱ ~20 min`

### Step 1 · Hang the second strip off one link

Chain strip 2 into one of its own frames, then attach that frame to strip 1 through **one**
accepted cross-strip pair. Below, 104 hangs off 20: $G_i = G_{20}\,H_{104\to20}\,G^{(2)}_i$,
where $G^{(2)}_i$ is strip 2's own chain into 104.

There are now **29** accepted pairs to check, and the joined mosaic used only **10** of them.

In [ ]:
def join(G1, link):
    # Attach STRIP2 to strip 1's mosaic through one cross-strip pair link = (a in strip 1, b in strip 2).
    a, b = link
    G2 = chain(STRIP2, b)                              # strip 2 into frame b
    Hb = G1[a] @ H_between(b, a)                       # frame b into the reference frame
    return {**G1, **{i: Hb @ g for i, g in G2.items()}}

try:
    G_link = join(G1, (IDX["16046-20"], IDX["16046-104"]))
    rms_link, E_link = summarise(G_link, "both strips, one link (20–104)")
    print()
    for a, b in [(0, 6), (1, 7), (2, 8), (3, 9), (4, 10)]:
        summarise(join(G1, (a, b)), f"   ... or link {NAMES[a][6:]}–{NAMES[b][6:]} instead")
except (NotImplementedError, NameError):
    print("Complete Tasks 2 and 3 first.")

Whichever link you choose, the pairs **far from it** disagree by 10–20 px. That is
35–70 m on the ground, well over a field boundary. Where is it, and what does it look like?
The left panel draws the **match graph**: one dot per frame, one line per accepted pair,
coloured by how badly that pair disagrees. Thick lines are the pairs the mosaic was built
from. The right panel is the mosaic at full resolution (×2), around the worst inlier of the
worst pair.

In [ ]:
ERR_CMAP = plt.matplotlib.colors.LinearSegmentedColormap.from_list("err", [TEAL, GOLD, RED, "#5A0F0F"])

def draw_graph(ax, G, used, title, vmax=15):
    # Match graph in mosaic coordinates: nodes = frame centres, edges = accepted pairs.
    E = pair_errors(G)
    ctr = {i: project(np.array([[FRAMES[i].shape[1] / 2, 500.0]]), G[i])[0] for i in G}
    segs = [[ctr[i], ctr[j]] for (i, j) in E]
    lw = [5 if ((i, j) in used or (j, i) in used) else 1.6 for (i, j) in E]
    lc = LineCollection(segs, cmap=ERR_CMAP, norm=plt.Normalize(0, vmax), linewidths=lw)
    lc.set_array(np.array([E[k][0] for k in E])); ax.add_collection(lc)
    for i, c in ctr.items():
        ax.scatter(*c, s=420, c="white", edgecolors=NAVY, zorder=3)
        ax.text(*c, NAMES[i][6:], ha="center", va="center", fontsize=9, zorder=4)
    ax.set_aspect("equal"); ax.invert_yaxis(); ax.axis("off"); ax.set_title(title)
    ax.margins(0.08)
    return lc

def worst_point(G, pair):
    # The inlier of `pair` that disagrees most under G, as a pixel of frame i.
    i, j = pair
    v = PAIRS[pair]
    e = np.linalg.norm(project(v["p"], G[i]) - project(v["q"], G[j]), axis=1)
    return v["p"][e.argmax()]

def hotspot(G, M, T, i, p, half=(200, 150)):
    # Crop of mosaic M around pixel p of frame i (wherever G puts it), enlarged x2.
    c = project(p[None], T @ G[i])[0].astype(int)
    x0 = int(np.clip(c[0] - half[0], 0, M.shape[1] - 2*half[0])); y0 = int(np.clip(c[1] - half[1], 0, M.shape[0] - 2*half[1]))
    crop = M[y0:y0 + 2*half[1], x0:x0 + 2*half[0]]
    return cv2.resize(crop, None, fx=2, fy=2, interpolation=cv2.INTER_NEAREST)

def tree_edges(order):
    # The consecutive pairs a chain along `order` uses.
    return {(order[k], order[k + 1]) for k in range(len(order) - 1)}

try:
    USED_LINK = tree_edges(STRIP1) | tree_edges(STRIP2) | {(IDX["16046-20"], IDX["16046-104"])}
    MOS_link, T_link = mosaic(G_link)
    WORST = max(E_link, key=lambda k: E_link[k][0])
    P_WORST = worst_point(G_link, WORST)                   # one fixed spot on the ground, reused below
    fig, ax = plt.subplots(1, 2, figsize=(16, 5.2), gridspec_kw={"width_ratios": [1.25, 1]})
    lc = draw_graph(ax[0], G_link, USED_LINK, "one link (20–104): pair disagreement, px RMS")
    plt.colorbar(lc, ax=ax[0], fraction=0.03, label="px RMS")
    ax[1].imshow(hotspot(G_link, MOS_link, T_link, WORST[0], P_WORST), cmap="gray"); ax[1].axis("off")
    ax[1].set_title(f"{NAMES[WORST[0]]} meets {NAMES[WORST[1]]} ({E_link[WORST][0]:.0f} px RMS): ghosts")
    plt.tight_layout(); plt.show()
except (NotImplementedError, NameError):
    print("Complete Tasks 2 and 3 first.")

### Step 2 · Use the strongest pairs? A maximum spanning tree

The mosaic needs exactly one path from each frame to the reference: a **spanning tree** of
the match graph. A natural idea: grow the tree from frame 20, each time adding the
not-yet-placed frame with the **most inliers** to a placed one (Prim's algorithm). Strong
pairs, short errors: surely better than our arbitrary choice of link?

In [ ]:
def max_spanning_tree(ref=REF):
    if not PAIRS:
        raise NotImplementedError("needs Task 1")
    G, used = {ref: np.eye(3)}, set()
    acc = [(k, v["n_inl"]) for k, v in PAIRS.items() if v["ok"]]
    while len(G) < len(NAMES):
        best = None
        for (i, j), n in acc:
            for a, b in ((i, j), (j, i)):                    # a placed, b not yet
                if a in G and b not in G and (best is None or n > best[0]):
                    best = (n, a, b)
        _, a, b = best
        G[b] = G[a] @ H_between(b, a); used.add((a, b))
    return G, used

try:
    G_tree, USED_TREE = max_spanning_tree()
    print("tree (frame <- parent):", ", ".join(f"{NAMES[b][6:]}<-{NAMES[a][6:]}" for a, b in sorted(USED_TREE, key=lambda e: e[1])))
    rms_tree, E_tree = summarise(G_tree, "maximum spanning tree")
    summarise(G_link, "one link (20–104), for comparison")
except (NotImplementedError, NameError):
    print("Complete Tasks 2 and 3 first.")

### Step 3 · Let every pair vote: global refinement  (Task 4)

Any tree uses 10 pairs and ignores the other 19. The fix is to **solve for all the
$G_i$ at once** so that *every* accepted pair agrees as well as possible:

$$\min_{G_i,\; i \ne 20}\;\; \sum_{(i,j)\ \text{accepted}}\;\; \sum_{p \leftrightarrow q}
\big\lVert G_i\,p - G_j\,q \big\rVert^2, \qquad G_{20} = I \text{ fixed.}$$

This is a small **bundle adjustment**: 10 homographies × 8 numbers = 80 unknowns, and about
11,000 residuals. `scipy.optimize.least_squares` solves it from any reasonable start (we use
the one-link mosaic) if you give it a function that returns **all the residuals as one
flat vector**. `unpack` turns the 80 numbers back into the dict of $G_i$ (with
$h_{33} = 1$).

In [ ]:
FREE = [k for k in range(len(NAMES)) if k != REF]           # the 10 frames we solve for
ACC = [k for k, v in PAIRS.items() if v["ok"]]              # the accepted pairs

def pack(G):
    return np.concatenate([(G[k] / G[k][2, 2]).ravel()[:8] for k in FREE])

def unpack(x):
    G = {REF: np.eye(3)}
    for t, k in enumerate(FREE):
        G[k] = np.append(x[8*t:8*t + 8], 1.0).reshape(3, 3)
    return G

In [ ]:
def residuals(x):
    '''
    x : the 80 free numbers. Returns one flat vector: for every accepted pair (i, j),
        the x and y differences between frame i's inliers and frame j's inliers, both
        projected into the mosaic.
    '''
    G = unpack(x)
    return np.concatenate([(project(PAIRS[k]["p"], G[k[0]]) - project(PAIRS[k]["q"], G[k[1]])).ravel()
                           for k in ACC])

In [ ]:
try:
    x0 = pack(G_link)
    r0 = residuals(x0)
    t0 = time.perf_counter()
    sol = least_squares(residuals, x0, x_scale="jac")
    G_ba = unpack(sol.x)
    print(f"{len(x0)} unknowns, {len(r0):,} residuals; solved in {time.perf_counter() - t0:.1f} s "
          f"({sol.nfev} evaluations)\n")
    rows = {}
    for label, G in (("one link (20–104)", G_link), ("maximum spanning tree", G_tree),
                     ("global refinement", G_ba)):
        rows[label] = summarise(G, label)[0]
    print(f"\n19 -> 20 against H_REF:  link {err_vs_ref(np.linalg.inv(G_link[REF]) @ G_link[IDX['16046-19']]):.2f} px,  "
          f"global {err_vs_ref(np.linalg.inv(G_ba[REF]) @ G_ba[IDX['16046-19']]):.2f} px")
except NotImplementedError:
    print("Complete Task 4 (and Tasks 2-3) first.")
except NameError:
    print("Run Steps 1 and 2 first.")

In [ ]:
try:
    MOS_tree, T_tree = mosaic(G_tree)
    MOS_ba, T_ba = mosaic(G_ba)
    fig, ax = plt.subplots(1, 3, figsize=(17, 3.4))
    for a, (G, used, title) in zip(ax, ((G_link, USED_LINK, "one link"),
                                        (G_tree, USED_TREE, "maximum spanning tree"),
                                        (G_ba, set(ACC), "global refinement (all 29 used)"))):
        lc = draw_graph(a, G, used, title)
    plt.colorbar(lc, ax=ax, fraction=0.015, label="px RMS"); plt.show()

    i0 = WORST[0]
    show([hotspot(G_link, MOS_link, T_link, i0, P_WORST), hotspot(G_tree, MOS_tree, T_tree, i0, P_WORST),
          hotspot(G_ba, MOS_ba, T_ba, i0, P_WORST)],
         [f"one link: {pair_errors(G_link)[WORST][0]:.1f} px", f"spanning tree: {pair_errors(G_tree)[WORST][0]:.1f} px",
          f"global: {pair_errors(G_ba)[WORST][0]:.1f} px"], figw=17, figh=4.6)

    fig, ax = plt.subplots(figsize=(16, 8.6))
    ax.imshow(MOS_ba, cmap="gray"); outline(ax, G_ba, T_ba, STRIP1, GOLD); outline(ax, G_ba, T_ba, STRIP2, TEAL)
    ax.axis("off"); ax.set_title(f"Eleven frames, two flight lines, globally refined: "
                                 f"{MOS_ba.shape[1]} × {MOS_ba.shape[0]} px ≈ "
                                 f"{MOS_ba.shape[1]*3.5/1000:.0f} × {MOS_ba.shape[0]*3.5/1000:.0f} km")
    plt.tight_layout(); plt.show()
except NameError:
    print("Complete Task 4 and run the cell above first.")

### Checkpoint 3

**Q3a.** The maximum spanning tree uses the *strongest* pairs, yet it is **worse**
than the arbitrary one-link mosaic. Trace the tree's path from 17 to 101 in the graph.
Why does "strongest" pick that shape, and what should a good tree optimise instead?

*Your answer:*

**Q3b.** Global refinement made 19 → 20 slightly *worse* against `H_REF`, while
fixing 17–101 by more than 10 px. Is that a bug? What is the optimiser trading, and why
can't it have both?

*Your answer:*

**Q3c.** Global refinement makes the mosaic agree **with itself**. Name one way
it can still be wrong **on the map**, and what you would add to the optimisation to fix
it. (Think about what $G_{20} = I$ fixes, and what the 2020 orthophoto could give you.)

*Your answer:*

---
## ★ Bonus — B1 · What does `cv2.Stitcher` do differently?

OpenCV's stitching module runs the same pipeline in two lines. It has two modes:

* `cv2.Stitcher_PANORAMA`: assumes a camera that only **rotates** (L8 §10), estimates a
  focal length and a rotation per photo, and projects onto a **sphere**;
* `cv2.Stitcher_SCANS`: assumes a **flat** subject and an **affine** model per photo
  (scanners, documents, and, roughly, air photos).

Both add three things our pipeline lacks: **exposure compensation** (gains per image),
**seam finding** (each output pixel comes from *one* photo, cut along a path where the photos
agree), and **multi-band blending** across the seam.

In [ ]:
colour = [cv2.cvtColor(f, cv2.COLOR_GRAY2BGR) for f in FRAMES]
results = {}
for name, mode in (("SCANS", cv2.Stitcher_SCANS), ("PANORAMA", cv2.Stitcher_PANORAMA)):
    st = cv2.Stitcher_create(mode)
    t0 = time.perf_counter()
    status, pano = st.stitch(colour)
    dt = time.perf_counter() - t0
    results[name] = pano if status == cv2.Stitcher_OK else None
    print(f"{name:9} status {status} ({'OK' if status == 0 else 'failed'})  {dt:4.1f} s  "
          + (f"{pano.shape[1]} x {pano.shape[0]}" if status == 0 else ""))
for name, pano in results.items():
    if pano is not None:
        plt.figure(figsize=(16, 8.4)); plt.imshow(pano[..., 0], cmap="gray"); plt.axis("off")
        plt.title(f"cv2.Stitcher, {name} mode"); plt.tight_layout(); plt.show()

**B1.** Compare the two Stitcher mosaics with yours: the ink annotations, the
brightness across frame edges, and the outline of the whole block. Which mode's assumption
matches a 1958 survey flight? Which of Stitcher's three extras would you add to your own
pipeline first, and why?

*Your answer:*

---
## Reflection — End of Lab

**R1.** List the steps of today's pipeline. For each, name the one parameter or
decision that you think matters most, and what goes wrong if you get it wrong.

*Your answer:*

**R2.** Brown & Lowe's acceptance test, a spanning tree, and global refinement all
appear in phone panorama apps. Which of the three would matter **least** for a 5-photo
phone panorama taken by turning in place, and which would matter **most** for a
200-frame mosaic of Kings County? Why?

*Your answer:*

**R3.** Your project probably needs images from two **eras** on one grid, not two
strips from one flight. Which of today's steps carry over unchanged, and which one breaks
first? *(L8 §11 (b): 1958 vs 1968 found 9 "inliers" out of 133.)*

*Your answer:*

---
## ✏️ Try at home

Not graded.

**A. Your own panorama.** Stand in one spot and take 4–6 overlapping phone photos, *turning*
on the spot (pivot the phone, don't walk). Resize them to about 1000 px wide, put them in
`FRAMES`, and run Parts 1–3 with the reference in the middle. Then try `Stitcher_PANORAMA`.
Where does your flat-plane mosaic stretch, and why (L8 §10, $x = f\tan\theta$)?

**B. Choose the reference automatically.** Pick the frame that minimises the largest number of hops to
any other frame in the match graph. Does it agree with your choice of 20?

**C. A robust loss.** Every inlier in `residuals` counts with its square. Try
`least_squares(..., loss="huber", f_scale=2.0)`. Does it change anything here? When would it?